# Lecture 1: GenAI Exercises

Welcome to this hands-on notebook! It belongs to the **Exercises** section of *Applied AI – Agentic AI – Lecture 1*. The slides explain each step; this notebook is where you build it.

There are **6 exercises** in this notebook:

| Exercise | Tasks |
|---|---|
| **Exercise 1: Simple LLM Setup** | 1.1 Imports and Configuration · 1.2 Simple LLM Call |
| **Exercise 2: Generate Time Schedule with LLM** | 2.1 Craft the Schedule Prompt · 2.2 Advanced LLM Call with JSON Output · 2.3 Extract Today's Classes from an Image |
| **Exercise 3: Parse LLM Response** | 3.1 Extract the JSON from the LLM Response · 3.2 Extract the First Entry |
| **Exercise 4: Google Calendar Authentication** | 4.1 Set Up Google Cloud and OAuth · 4.2 Authenticate with `get_calendar_service()` |
| **Exercise 5: Create Calendar Event** | 5.1 Build the Event Creation Function |
| **Exercise 6: Complete Integration Test** | 6.1 Create Your First Automated Calendar Event |

**Learning objectives**
By the end of this notebook, you will be able to:
- Call an LLM through a REST API (POST request)
- Write system and user prompts with variables, few-shot examples and chain of thought
- Extract structured data (JSON) from text and images
- Authenticate with the Google Calendar API (OAuth)
- Create calendar events programmatically and chain all steps into one workflow

**Prerequisites**
- Basic Python knowledge
- Google account for Calendar API setup
- VS Code with Python extension installed

Let's get started! 🚀

## Exercise 1: Simple LLM Setup 🤖

In this exercise you make your first LLM call: a single user prompt sent to Azure OpenAI in a POST request.

### Task 1.1: Imports and Configuration

Install and import the required libraries, then load your credentials from the `.env` file. Each of you gets an API key and an Azure OpenAI endpoint.

In [59]:
# Install required packages (run this first!)
# Uncomment the line below if you need to install packages
# !pip install requests google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("📦 Installing packages... (this may take a moment)")
print("✅ Installation complete! Now importing libraries...")

# Core Python libraries
import json
import datetime
import re
import os
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Get Google API credentials from environment variables
Client_id = os.getenv("GOOGLE_CLIENT_ID")
client_secret = os.getenv("GOOGLE_CLIENT_SECRET")

# Here the config file is initialized 
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_deployment_name = os.getenv("OPENAI_DEPLOYMENT_NAME")
openai_version_name = os.getenv("OPENAI_VERSION_NAME")

from pathlib import Path

# HTTP requests for API calls
import requests

# Google Calendar API libraries
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
import pickle

print("✅ All libraries imported successfully!")
print("🎯 Ready to start the exercises!")

📦 Installing packages... (this may take a moment)
✅ Installation complete! Now importing libraries...
✅ All libraries imported successfully!
🎯 Ready to start the exercises!


#### Check your Azure OpenAI configuration

Run the cell below to check that your credentials were loaded from `.env`.

**For Azure OpenAI users:**
- Get your endpoint and API key from Azure Portal
- Find your deployment name

**For OpenAI users:**
- Get your API key from https://platform.openai.com/

In [ ]:
# 🔧 Configuration 
print(azure_openai_endpoint)
print(azure_openai_api_key)
print(openai_deployment_name)
print(openai_version_name)

# 🔍 Choose your LLM provider
print("🔧 Configuration loaded!")

# ⚠️ Security reminder: Never commit API keys to version control!

### Task 1.2: Simple LLM Call

Ask a question (the **user prompt**) to an LLM within a **POST** request. This function will be the foundation for our schedule generation.
There is only a user prompt (no system prompt, no structured output).

Use the template below as a reference.

#### Reference: Basic LLM Call Template

Here's a very basic template for making LLM calls. This template shows the essential components you need:

##### 1. Basic Structure
```python
def simple_llm_call(prompt, api_endpoint = None, api_key = None, openai_deployment_name = None, openai_version_name = None):
    # Configuration
    api_url = f"{api_endpoint}/openai/deployments/{openai_deployment_name}/chat/completions?api-version={openai_version_name}"
    
    # Headers
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"  # or "api-key": api_key for Azure
    }
    
    # Payload
    payload = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 1000
    }
    
    # Make API call
    response = requests.post(api_url, headers=headers, json=payload)
    
    # Extract response
    result = response.json()
    return result["choices"][0]["message"]["content"]
```

##### 2. Key Components:
- **Endpoint**: The API URL where you send requests
- **Headers**: Authentication and content type
- **Payload**: Your prompt and parameters
- **Response Parsing**: Extract the actual text response

##### 3. Common Parameters:
- `temperature`: Controls randomness (0.0 = deterministic, 1.0 = creative)
- `max_tokens`: Maximum length of response
- `messages`: Array of conversation messages with roles (system, user, assistant)

This template works for most LLM APIs with minor adjustments!

In [ ]:
def simple_llm_call():
    """
    Simple function to call an LLM with a prompt
    
    Args:
        prompt (str): The prompt to send to the LLM
        max_tokens (int): Maximum tokens in response
    
    Returns:
        str: The LLM's response
    """
    
    # TODO: Students - Complete this function
    # 1. Create the API URL using the Azure OpenAI endpoint and deployment
    
    # 2. Set up the request headers with Content-Type and api-key

    
    # 3. Create the payload with messages, temperature, etc.

    
    # 4. Make the POST request
   
    
    # 5. Parse the response and return the content

    
    # Placeholder return - replace with actual implementation
    return "API does not work"
        
# Test the function
print("🧪 Testing LLM function...")
test_response = simple_llm_call("Hello! Can you respond with 'Hello World' to test the connection?", api_endpoint = None, api_key = None, openai_deployment_name = None, openai_version_name = None)
print(f"🤖 LLM Response: {test_response}")

🧪 Testing LLM function...
🤖 LLM Response: Hello World!
🤖 LLM Response: Hello World!


## Exercise 2: Generate Time Schedule with LLM 📅

Now let's build a more advanced LLM call that generates a daily schedule, and then use the LLM's vision capabilities to read a schedule from an image.

### Task 2.1: Craft the Schedule Prompt

Write a system prompt that creates a **non-overlapping schedule** for the given tasks on a certain date.
- **Variables:** `tasks` & `date` (filled into the user prompt)
- Use **few-shot prompting**: give the model solved examples
- Use **chain of thought**: ask the model to reason step by step before answering
- Ask for a **JSON** output format

In [62]:
schedule_system_prompt = """
You are an expert time management assistant. 
Your task is to generate a realistic and well-structured daily schedule for a student based on the given date and list of tasks.

Return the schedule as a JSON structure with the following fields: 
- date: The date for the schedule (e.g., "2024-10-01")
- start_time: The start time for the task (e.g., "09:00")
- end_time: The end time for the task (e.g., "10:00")
- task: one of the tasks from the provided list
- location: location of the task (e.g., "Office", "Home", "Library")
- type: type of task (e.g., "Study", "Meeting", "Break", "Exercise")

Requirements:
1. Use 24-hour time format (HH:MM)
2. Schedule time slots without overlap, covering the full day from 08:00 to 22:00
3. Include appropriate breaks (e.g., lunch, short rests)
4. Assign suitable locations for each task
5. Categorize each task with a proper type
6. Make realistic and balanced time allocations

Format the response strictly as JSON, following this structure:

{
    "{date}" : {
        "schedule": [
            {
                "start_time": "09:00",
                "end_time": "10:30",
                "task": "Study Session",
                "location": "Library",
                "type": "Study"
            },
            {
                "start_time": "14:00", 
                "end_time": "15:00",
                "task": "Team Meeting",
                "location": "Conference Room A",
                "type": "Meeting"
            }
        ]
    }
}

Only respond with valid JSON — do not include any explanations or text outside the JSON structure.
"""


schedule_user_prompt = """Generate a daily schedule for the date {date} based on the following tasks: {tasks}."""

### Task 2.2: Advanced LLM Call with JSON Output

Combine the components (system prompt, user prompt and variables) in a more advanced LLM call, and check that the structured output is valid JSON.

Use the template below as a reference.

#### Reference: Advanced LLM Call Template with Variables 🚀

Here's a more sophisticated template that includes system prompts, user prompts with variables, and advanced configuration options:

##### 1. Advanced Structure with System & User Prompts
```python
def advanced_llm_call(user_input, variables=None, system_prompt=None, api_endpoint=None, api_key=None, openai_deployment_name=None, openai_version_name=None):
    # Configuration
    api_url = f"{api_endpoint}/openai/deployments/{openai_deployment_name}/chat/completions?api-version={openai_version_name}"

    # Default system prompt if none provided
    if system_prompt is None:
        system_prompt = "You are a helpful AI assistant."

    # Format user input with variables if provided
    if variables:
        formatted_user_input = user_input.format(**variables)
    else:
        formatted_user_input = user_input

    # Headers
    headers = {
        "Content-Type": "application/json",
        "api-key": api_key  # Azure style
        # "Authorization": f"Bearer {api_key}"  # OpenAI style
    }

    # Advanced payload with system and user messages
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": formatted_user_input}
        ],
        "temperature": 0.1,      # Low for consistent results
        "max_tokens": 4000,      # Longer responses
        "top_p": 0.9,            # Nucleus sampling
        "frequency_penalty": 0,  # Reduce repetition
        "presence_penalty": 0    # Encourage new topics
    }

    # Make API call
    response = requests.post(api_url, headers=headers, json=payload, timeout=120)

    # Parse response
    result = response.json()
    return result["choices"][0]["message"]["content"]
```

##### 2. Key Advanced Features:
- **System Prompts**: Set the AI's behavior and role
- **Variable Substitution**: Dynamic content with `.format(**variables)`
- **Multi-message Conversations**: System + User + Assistant history
- **Advanced Parameters**:
  - `temperature`: Creativity level (0.0-1.0)
  - `top_p`: Alternative to temperature (0.0-1.0)
  - `max_tokens`: Response length limit
  - `frequency_penalty`: Reduce repetitive words
  - `presence_penalty`: Encourage topic diversity

##### 3. Usage Examples:
```python
# Example 1: Simple variable substitution
user_template = "Create a schedule for {date} with tasks: {tasks}"
variables = {"date": "2025-10-28", "tasks": "meeting, study, exercise"}

# Example 2: Complex system prompt
system_prompt = """You are an expert scheduler. Always respond in JSON format
with fields: date, start_time, end_time, activity, location, type."""

response = advanced_llm_call(user_template, variables, system_prompt)
```

In [ ]:
def llm_call_with_prompt_and_variables():
    """
    Function to call LLM with system prompt and formatted user prompt
    
    Args:
        date (str): The date for the schedule
        tasks (str): The tasks to schedule
    
    Returns:
        str: The LLM's response with JSON schedule
    """
    
    # TODO: Students - Complete this function
    # 1. Create the API URL
    
    # 2. Set up headers

    
    # 3. Format the user prompt with variables
    
    # 4. Create payload with system and user messages

    
    # 5. Make the API call and return content

    
    # Placeholder return - replace with actual implementation
    return '{"2025-10-28": {"schedule": [{"start_time": "09:00", "end_time": "10:30", "task": "Machine Learning lecture", "location": "Room A101", "type": "Study"}]}}'

In [ ]:
# 🧪 Test the prompt generation
# TODO: Students - Uncomment and test after implementing the function above
sample_tasks = ["Machine Learning lecture", "Python programming lab", "Team project meeting", "Study time"]
json_schedule = llm_call_with_prompt_and_variables(schedule_user_prompt, schedule_system_prompt, api_endpoint=azure_openai_endpoint, api_key=azure_openai_api_key, openai_deployment_name=openai_deployment_name, openai_version_name=openai_version_name, date="2025-10-28", tasks=sample_tasks)
print(json_schedule)

print("📝 TODO: Complete the llm_call_with_prompt_and_variables function above, then uncomment this test code.")

{
    "2025-10-28": {
        "schedule": [
            {
                "start_time": "08:00",
                "end_time": "09:30",
                "task": "Machine Learning lecture",
                "location": "Lecture Hall 3",
                "type": "Study"
            },
            {
                "start_time": "09:30",
                "end_time": "10:00",
                "task": "Morning Break",
                "location": "Cafeteria",
                "type": "Break"
            },
            {
                "start_time": "10:00",
                "end_time": "12:00",
                "task": "Python programming lab",
                "location": "Computer Lab 2",
                "type": "Study"
            },
            {
                "start_time": "12:00",
                "end_time": "13:00",
                "task": "Lunch Break",
                "location": "Cafeteria",
                "type": "Break"
            },
            {
                "start_time": "13:00",

### Task 2.3: Extract Today's Classes from an Image 📸

Use GPT-4o's vision capabilities to extract the courses from the schedule **`lessenrooster.png`** that are planned **today**. This demonstrates multimodal AI: the image is sent together with the text prompt.

The system and user prompts are given below. Use the vision template after them as a reference for your function.

In [ ]:
# 📸 Image Vision System and User Prompts for Event Extraction
import base64
from datetime import datetime, date



# System prompt for image-based event extraction
image_vision_system_prompt = """
You are an expert academic schedule extraction assistant with vision capabilities.
Your goal is to analyze images of class timetables and extract today's classes.

Current date: {today_date}
Current weekday: {today_weekday}
Current ISO week number: {today_week_number}

EXTRACTION RULES:
1. Carefully examine the image for any weekly schedule information.
2. Identify the column corresponding to today's weekday ({today_weekday}).
3. Extract all class entries for that day, including start and end times, subject names, and notes (e.g., 'even weeks', 'odd weeks', 'week 1-6', etc.).
4. If the class applies only to certain weeks, check whether {today_week_number} fits the condition:
   - If the note says "even weeks" and {today_week_number} is even → include it.
   - If the note says "odd weeks" and {today_week_number} is odd → include it.
   - If the note says "week X–Y" → include only if {today_week_number} is within that range.
5. Ignore all other days and weeks not matching today's conditions.
6. If no classes apply today, return an empty schedule.

RESPONSE FORMAT:
Return the extracted events in this JSON structure:
{{
    "{today_date}": {{
        "schedule": [
            {{
                "start_time": "HH:MM",
                "end_time": "HH:MM",
                "activity": "Course name from image",
                "location": "Location if visible, else null",
                "type": "Categorize as 'Study'"
            }}
        ]
    }}
}}

ADDITIONAL RULES:
- Use 24-hour time format (HH:MM).
- Merge cells or notes (like "PC-oefeningen" or "practicum") into the same activity field if relevant.
- Translate Dutch day or week labels (e.g. "even weken", "oneven weken") into English meaning when applying logic.
- If no classes for today are found, return:
  {{"{today_date}": {{"schedule": []}}}}
- Return only valid JSON output — no extra explanations, no markdown.
"""


# User prompt for image analysis
image_vision_user_prompt = """
Please analyze the attached image of a weekly class timetable and extract all lessons or events that occur today ({today_date}),
which is a {today_weekday} in ISO week {today_week_number}.

Instructions:
- Identify the column corresponding to {today_weekday}.
- Extract all lessons listed for that day, including:
  - Subject or course name
  - Start and end time
  - Notes such as "even weeks", "odd weeks", or "week ranges"
  - Type (e.g., theory, exercises, practicum)
- If a note restricts the class to certain weeks, only include it if {today_week_number} matches the condition.
  - Example: include "even weeks" only if this is an even week.
- If no classes apply today, return an empty schedule.

Output requirements:
- Use the JSON format specified in the system prompt
- Use 24-hour time format
- Do not add explanations or text outside the JSON output
"""


print("✅ Image vision prompts created!")
print("🎯 Ready to process lessenrooster.png for today's events!")

✅ Image vision prompts created!
🎯 Ready to process download.jpeg for today's events!


#### Reference: Vision Template with Image Processing

Here's a template for working with GPT-4o vision capabilities to process images:

##### 1. Vision API Structure with Base64 Encoding
```python
def extract_events_from_image(image_path, user_prompt, system_prompt, variables, api_endpoint=None, api_key=None, openai_deployment_name=None, openai_version_name=None):
    try:
        # Step 1: Read and encode image to base64
        with open(image_path, "rb") as image_file:
            image_data = base64.b64encode(image_file.read()).decode('utf-8')
        
        # Step 2: Create API endpoint for vision
        api_url = f"{api_endpoint}/openai/deployments/{openai_deployment_name}/chat/completions?api-version={openai_version_name}"
        
        # Step 3: Set up headers (same as text API)
        headers = {
            "Content-Type": "application/json",
            "api-key": api_key
        }
        
        # Step 4: Format prompts with variables
        formatted_system_prompt = system_prompt.format(
            **variables
        )
        formatted_user_prompt = user_prompt.format(
            **variables
        )
        
        # Step 5: Create payload with image and text
        payload = {
            "messages": [
                {"role": "system", "content": formatted_system_prompt},
                {
                    "role": "user", 
                    "content": [
                        {
                            "type": "text",
                            "text": formatted_user_prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{image_data}"
                            }
                        }
                    ]
                }
            ],
            "temperature": 0.1,
            "max_tokens": 4000
        }
        
        # Step 6: Make API call with longer timeout
        response = requests.post(api_url, headers=headers, json=payload, timeout=120)
        
        # Step 7: Parse and return response
        response_data = response.json()
        return response_data["choices"][0]["message"]["content"]
        
    except Exception as e:
        return f'Error: {str(e)}'
```

##### 2. Key Vision API Features:
- **Base64 Encoding**: Images must be converted to base64 strings
- **Multi-modal Content**: User message contains both text and image_url
- **Image URL Format**: `data:image/jpeg;base64,{base64_string}`
- **Longer Timeout**: Vision processing takes more time (120 seconds recommended)
- **Variable Substitution**: Format prompts with dynamic date/time information

##### 3. Content Structure for Vision:
```python
# For vision API, user content is an array with different types:
"content": [
    {
        "type": "text",
        "text": "Your text prompt here"
    },
    {
        "type": "image_url", 
        "image_url": {
            "url": "data:image/jpeg;base64,{your_base64_image}"
        }
    }
]
```

##### 4. Common Vision Use Cases:
- **Schedule Extraction**: Read timetables and extract events
- **Document Analysis**: Parse forms, receipts, or structured documents  
- **Academic Calendar Processing**: Extract class schedules with week-specific rules
- **Visual Data Entry**: Convert image information to structured JSON

This template enables powerful multimodal AI capabilities!

In [ ]:
def extract_events_from_image():
    """
    Extract events from an image using GPT-4o vision capabilities
    
    Args:
        image_path (str): Path to the image file
        today_date (str): Today's date
        today_weekday (str): Today's weekday
        today_week_number (int): Today's week number
        
    Returns:
        str: JSON response with extracted events for today's date
    """
    
    try:
        # TODO: Students - Complete this function
        # 1. Read and encode the image to base64
        # Hint: Use base64.b64encode() and decode to 'utf-8'
        
        
        # 2. Create the API URL for Azure OpenAI with vision
        # Hint: Use the same pattern as other API calls
        
        
        # 3. Set up headers
        # Hint: Same headers as other API calls
        
        
        # 4. Format the system and user prompts with variables
        # Hint: Use .format() method with the date/weekday parameters
        
                
        # Step 4: Format prompts with variables

        # 5. Create payload with both text and image content
        # The payload should include:
        # - system message with formatted_system_prompt
        # - user message with both text and image_url content
        # - temperature and max_tokens
        
        
        # 6. Make the API call and return content
        # Hint: Use requests.post() with timeout=120
        
        
        # Placeholder return - replace with actual implementation
        return 'failed'
        
    except FileNotFoundError:
        return f'Error: ...Image file not found.'
    except Exception as e:
        return f'Error processing image: {str(e)}'

# Test the image extraction function - Students should complete function first



🔍 Extracting events from lessenrooster.png
📸 GPT-4o Vision Response:
{
    "2025-10-30": {
        "schedule": [
            {
                "start_time": "08:15",
                "end_time": "10:15",
                "activity": "Calculus oefeningen",
                "location": null,
                "type": "Study"
            },
            {
                "start_time": "10:30",
                "end_time": "12:30",
                "activity": "Mechanica, trillingen en golven practicum",
                "location": null,
                "type": "Study"
            },
            {
                "start_time": "13:30",
                "end_time": "15:30",
                "activity": "Dierkunde practicum",
                "location": null,
                "type": "Study"
            }
        ]
    }
}
📸 GPT-4o Vision Response:
{
    "2025-10-30": {
        "schedule": [
            {
                "start_time": "08:15",
                "end_time": "10:15",
                "activ

In [ ]:
#📝 TODO: Complete the extract_events_from_image function above first"
print("💡 After implementation, uncomment the test code below:")
# Get today's date for filtering events"
today = date.today()
today_date = today.isoformat()
today_weekday = today.strftime('%A')
today_week_number = today.isocalendar()[1]
#TEST CODE - Uncomment after implementation
image_events_response = extract_events_from_image("lessenrooster.png", image_vision_user_prompt, image_vision_system_prompt, today_date, today_weekday, today_week_number, azure_openai_endpoint, azure_openai_api_key, openai_deployment_name, openai_version_name)
# print(f"🤖 Extracted Events Response: {image_events_response}")

## Exercise 3: Parse LLM Response 🔍

The LLM gives us a schedule, but as **text**. To use it with the Google Calendar API we first need structured data.

### Task 3.1: Extract the JSON from the LLM Response

Turn the LLM's text into a Python dictionary: find the `{ ... }` block with a regex, then parse it with `json.loads()`.

In [ ]:
image_events_response = """{
    "2025-10-30": {
        "schedule": [
            {
                "start_time": "08:15",
                "end_time": "10:15",
                "activity": "Calculus oefeningen",
                "location": null,
                "type": "Study"
            },
            {
                "start_time": "10:30",
                "end_time": "12:30",
                "activity": "Mechanica, trillingen en golven practicum",
                "location": null,
                "type": "Study"
            },
            {
                "start_time": "13:30",
                "end_time": "15:30",
                "activity": "Dierkunde practicum",
                "location": null,
                "type": "Study"
            }
        ]
    }
}"""

In [67]:
def extract_json_schedule(llm_response):
    """
    Extract JSON schedule from LLM response string.
    
    Args:
        llm_response (str): The raw response from the LLM.
    
    Returns:
        dict: Parsed JSON schedule.
    """
    try:
        # Use regex to find the JSON part of the response
        json_match = re.search(r'\{.*\}', llm_response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            schedule = json.loads(json_str)
            return schedule
        else:
            raise ValueError("No JSON found in LLM response.")
    except json.JSONDecodeError as e:
        raise ValueError(f"Error decoding JSON: {e}")
    
# Extract and print the structured schedule
structured_schedule = extract_json_schedule(image_events_response)
print("📅 Structured Schedule:")
print(structured_schedule)

📅 Structured Schedule:
{'2025-10-30': {'schedule': [{'start_time': '08:15', 'end_time': '10:15', 'activity': 'Calculus oefeningen', 'location': None, 'type': 'Study'}, {'start_time': '10:30', 'end_time': '12:30', 'activity': 'Mechanica, trillingen en golven practicum', 'location': None, 'type': 'Study'}, {'start_time': '13:30', 'end_time': '15:30', 'activity': 'Dierkunde practicum', 'location': None, 'type': 'Study'}]}}


### Task 3.2: Extract the First Entry

Take the first item of the schedule and add the date to it. This entry becomes the calendar event in Exercise 5.

In [58]:
def extract_first_entry(parsed_schedule, date):
    """
    Extract the first schedule entry and format it for calendar creation
    
    Args:
        parsed_schedule (dict): Parsed schedule data
        date (str): The date for the schedule
        
    Returns:
        dict: First calendar entry or None
    """

    first_entry = parsed_schedule[date]["schedule"][0].copy()  # Create a copy to avoid modifying original
    first_entry['date'] = date  # Add the date to the entry
    return first_entry
first_entry = extract_first_entry(structured_schedule, "2025-10-28")
print(first_entry)

{'start_time': '08:15', 'end_time': '10:15', 'activity': 'Plantkunde theorie', 'location': None, 'type': 'Study', 'date': '2025-10-28'}


## Exercise 4: Google Calendar Authentication 🔐

Before we can schedule events, we need access to your Google Calendar. We use **OAuth with a Desktop app client**: you log in once in the browser, and the notebook gets a token to act on your calendar.

### Task 4.1: Set Up Google Cloud and OAuth 🔧

Follow these steps (they match Steps 1–5 on the slides).

#### Step 1: Go to Google Cloud & create a project
1. Go to [Google Cloud Console](https://console.cloud.google.com/) and log in
2. Click on the project picker to select a project or create a new one
3. Click **New project**

#### Step 2: Create the project
1. Call the project **"Howest-usecase"**
2. Set up 2-factor authentication if asked (this can be with your phone number)

#### Step 3: Enable the Google Calendar API
1. Search for **"Google Calendar API"** in the search bar
2. Click **Enable**

#### Step 4: Authentication
1. Go to **APIs & Services** > **Credentials**
2. If the consent screen is not set up yet, click the link and fill in the app registration (**audience: External**)
3. Click **Create credentials** > **OAuth client ID**
4. Select **Desktop app** as the application type
5. You now have a **client ID** and **client secret**. Add them to your `.env` file:
   ```
   GOOGLE_CLIENT_ID=...
   GOOGLE_CLIENT_SECRET=...
   ```
6. Restart the kernel and rerun the imports cell (Task 1.1) so the new values are loaded

#### Step 5: Oh no – add yourself as a test user
If `get_calendar_service()` does not work out (e.g. "access blocked"):
1. Go to the consent screen setup from before > **Audience**
2. Under **Test users**, add your own email address

#### ⚠️ Security Note
Never share or publish your client ID and client secret. Never commit your `.env` file or `token.pickle` to version control! Add them to your `.gitignore`.

### Task 4.2: Authenticate with `get_calendar_service()`

Run the cells below. A browser window opens: log in with the test user from Step 5 and allow access to your calendar.
The token is saved in `token.pickle`, so you only have to log in once. If you change `SCOPES` (or the token expires), delete `token.pickle` and run the cell again.

In [ ]:
# Get Google API credentials from environment variables
Client_id = os.getenv("GOOGLE_CLIENT_ID")
client_secret = os.getenv("GOOGLE_CLIENT_SECRET")

# If modifying these scopes, delete the file token.pickle.

SCOPES = ['https://www.googleapis.com/auth/calendar']

def get_calendar_service():
    creds = None
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_config(
                {
                    "installed": {
                        "client_id": Client_id,
                        "client_secret": client_secret,
                        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
                        "token_uri": "https://oauth2.googleapis.com/token"
                    }
                },
                SCOPES
            )
            creds = flow.run_local_server(port=0)  # automatically opens browser and handles redirect

        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    service = build('calendar', 'v3', credentials=creds)
    return service

In [69]:
get_calendar_service()

## Exercise 5: Create Calendar Event 📅

Now we can create a real calendar event. This takes our parsed schedule entry (Task 3.2) and sends it to the Google Calendar API in a POST request.
A calendar event has a specific structure: see the [Google Calendar API documentation for `events.insert`](https://developers.google.com/workspace/calendar/api/v3/reference/events/insert).

### Task 5.1: Build the Event Creation Function

In [ ]:
def create_calendar_event(event_data):
    """
    Create a Google Calendar event from parsed schedule data
    
    Args:
        event_data (dict): Event information with fields like:
            - date: Event date (YYYY-MM-DD)
            - start_time: Start time (HH:MM)
            - end_time: End time (HH:MM)
            - activity: Event title/name
            - location: Event location (optional)
            - type: Event type (optional)
        
    Returns:
        dict: Result of event creation
    """
    
    try:
        # Get calendar service
        service = get_calendar_service()
        
        if not service:
            return {"success": False, "error": "No calendar service available"}
        
        # Parse date and time
        event_date = event_data["date"]
        start_time = event_data["start_time"]
        end_time = event_data["end_time"]
        
        # Create datetime strings (ISO format for Google Calendar)
        start_datetime = f"{event_date}T{start_time}:00"
        end_datetime = f"{event_date}T{end_time}:00"
        
        # Get event title - try different possible field names
        event_title = event_data.get("activity") or event_data.get("task") or "Scheduled Event"
        event_type = event_data.get("type", "")
        
        # Build the event structure
        event = {
            'summary': f"{event_title}" + (f" - {event_type}" if event_type else ""),
            'location': event_data.get("location", ""),
            'description': event_data.get("description", f"Auto-scheduled: {event_title}"),
            'start': {
                'dateTime': start_datetime,
                'timeZone': 'Europe/Brussels',  # Adjust for your timezone
            },
            'end': {
                'dateTime': end_datetime,
                'timeZone': 'Europe/Brussels',
            },
            'reminders': {
                'useDefault': False,
                'overrides': [
                    {'method': 'popup', 'minutes': 15},
                    {'method': 'email', 'minutes': 60},
                ],
            },
        }
        
        # Create the event
        print(f"📅 Creating event: {event_title}")
        print(f"🕒 Time: {start_time} - {end_time} on {event_date}")
        
        created_event = service.events().insert(calendarId='primary', body=event).execute()
        
        return {
            "success": True,
            "event_id": created_event.get('id'),
            "event_link": created_event.get('htmlLink'),
            "message": f"Successfully created event: {event_title}",
            "event_details": {
                "title": event_title,
                "date": event_date,
                "start_time": start_time,
                "end_time": end_time,
                "location": event_data.get("location", "")
            }
        }
        
    except Exception as e:
        return {
            "success": False,
            "error": f"Failed to create event: {str(e)}"
        }

# 🧪 Test function (we'll create the actual event in the next section)
print("✅ Calendar event creation function ready!")
print("🎯 We'll use this to schedule our first LLM-generated event next!")

✅ Calendar event creation function ready!
🎯 We'll use this to schedule our first LLM-generated event next!


## Exercise 6: Complete Integration Test 🎯

This is the exciting part! Let's bring everything together into one workflow:

1. ✅ Use the LLM to extract today's classes from the image (Task 2.3)
2. ✅ Parse the response into JSON (Task 3.1)
3. ✅ Extract the first entry (Task 3.2)
4. ✅ Create a Google Calendar event (Task 5.1)

### Task 6.1: Create Your First Automated Calendar Event

Replace the hardcoded `first_entry` below with the output of the steps above.

**Important**: This will create a REAL event in your Google Calendar!

In [73]:
first_entry = {'start_time': '08:15', 'end_time': '10:15', 'activity': 'Plantkunde theorie', 'location': None, 'type': 'Study', 'date': '2025-10-28'}
event_result = create_calendar_event(first_entry)
print(event_result)

📅 Creating event: Plantkunde theorie
🕒 Time: 08:15 - 10:15 on 2025-10-28
{'success': True, 'event_id': 't5effovahnqidhd7vfiuq59vco', 'event_link': 'https://www.google.com/calendar/event?eid=dDVlZmZvdmFobnFpZGhkN3ZmaXVxNTl2Y28ganVsaWV2YW5hY2tlcmUxNEBt', 'message': 'Successfully created event: Plantkunde theorie', 'event_details': {'title': 'Plantkunde theorie', 'date': '2025-10-28', 'start_time': '08:15', 'end_time': '10:15', 'location': None}}
{'success': True, 'event_id': 't5effovahnqidhd7vfiuq59vco', 'event_link': 'https://www.google.com/calendar/event?eid=dDVlZmZvdmFobnFpZGhkN3ZmaXVxNTl2Y28ganVsaWV2YW5hY2tlcmUxNEBt', 'message': 'Successfully created event: Plantkunde theorie', 'event_details': {'title': 'Plantkunde theorie', 'date': '2025-10-28', 'start_time': '08:15', 'end_time': '10:15', 'location': None}}
